In [1]:
!pip install -q -U transformers evaluate rouge_score gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.9 MB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr
import evaluate

# ---------- 1. Load BART model ----------

model_name = "facebook/bart-large-cnn"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")


# ---------- 2. Summarization function ----------

def summarize_text(input_text):

    if not input_text.strip():
        return "Please enter some text."

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    summary_ids = model.generate(
        **inputs,
        max_length=45,
        min_length=15,
        do_sample=False
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


# ---------- 3. Test the model ----------

test_text = """
Artificial intelligence is transforming many industries.
Generative AI can create text, images, audio and computer code.
These technologies are used in education, healthcare, business,
software development and many other fields.
"""

print("\nGenerated Summary:")
print(summarize_text(test_text))


# ---------- 4. ROUGE Evaluation ----------

rouge = evaluate.load("rouge")

generated_summaries = [
    "AI models generate new content such as text and images."
]

reference_summaries = [
    "Generative AI models are capable of producing new content including text and images."
]

scores = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

print("\nROUGE Evaluation Scores:")
print(scores)


# ---------- 5. Create Gradio App ----------

demo = gr.Interface(
    fn=summarize_text,
    inputs=gr.Textbox(
        lines=8,
        label="Enter text to summarize"
    ),
    outputs=gr.Textbox(
        label="Generated Summary"
    ),
    title="GenAI Text Summarizer",
    description="A cloud-deployable Generative AI summarization app built with Gradio."
)


# ---------- 6. Launch ----------

demo.launch(share=True)

Loading model...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

...

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully!

Generated Summary:
Automation is transforming many industries. Generative AI can create text, images, audio and computer code.



ROUGE Evaluation Scores:
{'rouge1': np.float64(0.608695652173913), 'rouge2': np.float64(0.380952380952381), 'rougeL': np.float64(0.608695652173913), 'rougeLsum': np.float64(0.608695652173913)}
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6f771279a76756b086.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
